In [1]:
import numpy as np
import matplotlib.pyplot as plt
import numba

from molsim import mullerBrownPotential, mullerBrownPotentialAndGradient, plot_muller_brown_heatmap

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

# Exercise 2: Implementing a Langevin Integrator for NVT Simulations

In the previous task, you implemented a **velocity Verlet integrator** for the **NVE ensemble**, where total energy is conserved. This task extends to the **NVT ensemble**, where temperature is controlled using a **Langevin integrator**.

The Langevin equation, introduced by Paul Langevin in 1908 [1], describes the stochastic motion of particles in a viscous fluid. For molecular dynamics, it balances deterministic forces, friction, and stochastic noise:
$$
m \frac{d^2\mathbf{r}}{dt^2} = -\gamma m \frac{d\mathbf{r}}{dt} - \nabla U(\mathbf{r}) + \mathbf{\eta}(t),
$$
where:
- $m$ is the particle's mass (set to 1),
- $-\gamma m \frac{d\mathbf{r}}{dt}$ is the friction term,
- $-\nabla U(\mathbf{r})$ is the deterministic force,
- $\mathbf{\eta}(t)$ is the random force with $\langle \mathbf{\eta}(t) \rangle = 0$ and variance proportional to $T$.

This integrator introduces frictional damping and thermal fluctuations, enabling simulations at a fixed temperature. However, as noted by Frenkel and Smith, Langevin dynamics does not conserve momentum or include hydrodynamic interactions but remains effective for thermal equilibrium.

### Velocity update
We can wrap these equations of motion by rescaling the force with a random term. First, we calculate a decay function for the friction coefficient, setting

$
\theta = \exp(-\gamma \Delta t)
$.

The random process $\mathbf{\eta}(t)$ has a magnitude related to this decay with

$
\mathbf{\sigma}(t) = \sqrt{\left(1 - \theta^2 \right) T}
$

and

$
\mathbf{\eta}(t) = \sigma \mathcal{N}(0, 1)
$

A new rescaling of the velocity can be added, where 

$
v'\left(t + \frac{\Delta t}{2} \right) = \theta v\left(t + \frac{\Delta t}{2} \right) + \mathbf{\eta}(t)
$

### Question 1

Building on your velocity Verlet integrator, implement the Langevin integrator.
- **Iterate updates:** Adjust positions and velocities at each step.

In [ ]:
@numba.njit
def langevin(numberOfCycles: int, temperature: float, timeStep: float = 5e-4, seed: int = 0):
    """
    Implements the Langevin integrator to simulate the motion of a particle
    in a 2D potential field (Müller-Brown potential) within the NVT ensemble.

    Parameters:
        numberOfCycles (int): Total number of integration steps.
        temperature (float): Temperature of the system (canonical ensemble).
        timeStep (float): Time step for integration (default is 5e-4).
        seed (int): Seed for random number generation (default is 0, no seed).

    Returns:
        positions (np.ndarray): Array storing particle positions at each step.
        energies (np.ndarray): Array storing total energy (potential + kinetic) at each step.
    """
    # Set a random seed for reproducibility if specified
    if seed != 0:
        np.random.seed(seed)

    # Initialize arrays to store positions and energies
    positions = np.zeros((numberOfCycles, 2), dtype=np.float64)  # Positions: (steps, x/y coordinates)
    energies = np.zeros((numberOfCycles, 3), dtype=np.float64)  # Energies: total, potential, and kinetic

    # Set the initial position of the particle
    positions[0] = np.array([-0.557114228, 1.44889779])

    # Compute the initial potential energy and force at the starting position
    potentialEnergy, gradient_dx, gradient_dy = mullerBrownPotentialAndGradient(positions[0])
    force = -np.array([gradient_dx, gradient_dy])  # Force is the negative gradient of the potential

    # Initialize velocity with a random direction
    theta = 2 * np.pi * np.random.rand()  # Random angle in radians
    velocity = np.array([np.cos(theta), np.sin(theta)], dtype=np.float64)  # Unit vector in random direction

    # Scale the initial velocity to match the desired temperature
    velocity *= np.sqrt(2 * temperature)

    # Define damping coefficient (γ) and precompute factors for efficiency
    gamma = 1.0  # Friction coefficient
    theta = np.exp(-gamma * timeStep)  # Exponential decay factor for velocity
    sigma = np.sqrt((1 - theta**2) * temperature)  # Standard deviation of stochastic noise

    # Lambda function to compute the kinetic energy of the particle
    computeKineticEnergy = lambda v: 0.5 * np.sum(v**2)

    # Store initial energies
    energies[0, 0] = potentialEnergy + computeKineticEnergy(velocity)  # Total energy
    energies[0, 1] = potentialEnergy  # Potential energy
    energies[0, 2] = computeKineticEnergy(velocity)  # Kinetic energy

    # Main loop to integrate over the number of cycles
    for cycle in range(1, numberOfCycles):
        # Update velocity using half-step of the deterministic force
        velocity += 0.5 * force * timeStep

        # Apply damping and stochastic noise to the velocity
        # start refactor
        velocity = velocity
        # end refactor

        # Update position using the modified velocity
        positions[cycle] = positions[cycle - 1] + velocity * timeStep

        # Recompute potential energy and force at the new position
        potentialEnergy, gradient_dx, gradient_dy = mullerBrownPotentialAndGradient(positions[cycle])
        force = -np.array([gradient_dx, gradient_dy])  # Update force using the new position

        # Complete the velocity update with the new force
        velocity += 0.5 * force * timeStep

        # Compute and store energies
        energies[cycle, 0] = potentialEnergy + computeKineticEnergy(velocity)  # Total energy
        energies[cycle, 1] = potentialEnergy  # Potential energy
        energies[cycle, 2] = computeKineticEnergy(velocity)  # Kinetic energy

    # Return arrays containing positions and energies
    return positions, energies

In [ ]:
# Run simulation
timeStep = 5e-4
pos, energies = langevin(50000, 10, seed=50, timeStep=timeStep)

# Plot energy evolution and trajectory
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(np.arange(len(energies)) * timeStep, energies[:, 0], label="Total")
# ax1.plot(np.arange(len(energies)) * timeStep, energies[:,1], label="Potential")
# ax1.plot(np.arange(len(energies)) * timeStep, energies[:,2], label="Kinetic")
ax1.set_title("Energy Evolution")
ax1.set_xlabel(r"Time / $\tau$")
ax1.set_ylabel(r"Energy / $\varepsilon$")

plot_muller_brown_heatmap(ax2)
ax2.plot(*pos[::10].T, lw=0.5, c="red")
ax2.set_title("Trajectory")
ax2.set_xlabel("X position")
ax2.set_ylabel("Y position")

fig.tight_layout()
plt.show()

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

### Question 2
Run the Langevin integrator simulation to generate **8 trajectories**, each with a different temperature. 
Plot the trajectories on a Muller-Brown potential heatmap, arranging the plots in a \( 2 $\times$ 4 \) grid. 
As temperature increases, the system gains more energy to traverse energy barriers in the potential landscape.
 Compare how lower temperatures confine trajectories to local minima, while higher temperatures enable 
 exploration across multiple basins. Observe which regions are most frequently visited at different temperatures.

In [ ]:
timeStep = 5e-4
numSteps = int(1e6)
Temperatures = [0, 1, 5, 10, 15, 20, 25, 30]
nRuns = 1

# Plot the trajectories for each temperature
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes = axes.flatten()
colors = [
    "red",
    "cyan",
    "magenta",
    "lime",
    "orange",
    "darkviolet",
    "pink",
    "black",
]
for i, Temperature in enumerate(Temperatures):
    plot_muller_brown_heatmap(axes[i])
    for j in range(nRuns):
        pos, _ = langevin(numSteps, Temperature, timeStep=timeStep)
        axes[i].plot(*pos[::10].T, lw=0.5, alpha=0.4, c=colors[j % len(colors)])
        axes[i].set_title(f"Temperature = {Temperature}")

fig.tight_layout()
plt.show()

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">


### Question 3
Again analyze the speed of the particle. Plot a distribution of the speed and compare it to the Maxwell-Boltzmann distribution. Does the system follow the Maxwell-Boltzmann distribution?

In [ ]:
# Simulate Langevin for 1 million steps
_, energies = langevin(int(1e6), Temperature, timeStep=5e-4, seed=50)

Temperature = np.mean(energies[:, 2])  # Temperature from Kinetic energy (in 2D <E_K>= T)
beta = 1 / Temperature  # Reduced units (k_B = 1)

# Compute speeds from kinetic energies
speeds = np.sqrt(2 * energies[:, 2])

# Generate theoretical Maxwell-Boltzmann distribution
v = np.linspace(0, np.max(speeds), 500)
maxwellBoltzmannDistribition = beta * v * np.exp(-0.5 * beta * v**2)  # Maxwell-Boltzmann distribution for a 2d system

# Plot simulated vs theoretical
plt.figure(figsize=(8, 5))
plt.hist(speeds, bins=200, density=True, label="Simulated")
plt.plot(v, maxwellBoltzmannDistribition, "r--", label="Maxwell-Boltzmann (2D)")
plt.xlabel("Velocity")
plt.ylabel("Probability Density")
plt.title("Maxwell-Boltzmann Distribution in 2D")
plt.legend()
plt.show()

<div style="max-width: 1000px; margin-left: 0; margin-right: auto; font-size: 20px; line-height: 1.6;">

### Question 4  
Compare the velocity Verlet and Langevin integrators. What are the key differences in their underlying 
principles and how they handle system dynamics? Discuss the scenarios where each integrator is most appropriate:
- When would you prefer using the velocity Verlet integrator?
- When would the Langevin integrator be more suitable?  